# 1. Saúde reprodutiva e planejamento familiar das mulheres

# 2. Membros
Antônio Alberto Pereira de Oliveira - 2021086504

Kauan Valente - 2023060464

Maria Eduarda Sampaio - 2022055831

# 3. Descrição dos dados

**URL**
https://www.kaggle.com/datasets/konradb/wellbeing-of-women-in-52-countries

Para reproduzir: baixar o dataset no Kaggle e colocar os arquivos `livwell.csv` e `indicators.csv` em `data/raw/`.

**Domínio**
Saúde reprodutiva e planejamento familiar das mulheres e sua relação com escolaridade, trabalho, autonomia nas decisões e violência doméstica. As consultas concentram-se em três áreas temáticas: Saúde reprodutiva, Educação e trabalho, e Autonomia e violência. Os demais indicadores selecionados (demografia, domicílio e condições de vida, saúde infantil e nutrição) permanecem no banco como contexto para análises futuras. Os indicadores vêm de pesquisas DHS (*Demographic and Health Surveys*) e permitem comparar países, regiões e períodos.

**Processamento**

O conjunto de dados LivWell é composto originalmente por 1.832 registros e 409 colunas (`livwell.csv`), além de um dicionário com 265 indicadores (`indicators.csv`). Cada registro corresponde a uma região de um país em uma pesquisa DHS, e cada indicador ocupa uma coluna (com uma coluna `_se` adicional para o erro-padrão, quando disponível).

O processamento, implementado nas células abaixo, segue as etapas:

1. **Seleção de indicadores** — foram selecionados manualmente 100 indicadores relacionados ao domínio do trabalho, distribuídos em 16 categorias. A seleção está em `data/processed/filtered_indicators.csv`.
2. **Filtragem de `livwell.csv`** — foram mantidas apenas as colunas de identificação (país, região, pesquisa, data da entrevista) e as colunas de valor e erro-padrão dos 100 indicadores. Nenhum registro foi removido.
3. **Normalização** — os dados foram decompostos em sete relações: `Pais`, `Regiao`, `Pesquisa`, `ColetaPesquisaRegiao`, `CategoriaIndicador`, `Indicador` e `Medicao`. As colunas de indicadores foram convertidas em linhas (uma medição por pesquisa × região × indicador), removendo apenas as medições sem valor observado e mantendo erros-padrão ausentes como nulos.
4. **Carga e verificação** — as relações foram inseridas em um banco SQLite com chaves primárias, estrangeiras e restrições de unicidade. A validação de chaves estrangeiras é desativada durante a carga (`PRAGMA foreign_keys = 0`) e a integridade referencial é verificada ao final.

**Ajustes nos dados originais**

- **País**: extraído como entidade própria. Além do nome, recebeu o continente, atribuído pelo grupo (o dataset não possui essa informação). Turquia e Armênia, países transcontinentais, foram classificados na Ásia.
- **Data da entrevista**: `interview_year_mean`, `interview_month_mean` e `CMC_interview_mean` variam entre as regiões de uma mesma pesquisa. Por isso pertencem ao relacionamento `ColetaPesquisaRegiao` (pesquisa × região), e não à pesquisa. Em `Pesquisa`, `CMC_interview_mean` é mantido como atributo derivado (média das regiões).
- **Categorias**: no dataset original a categoria é apenas um texto repetido em cada indicador. Ela foi extraída para a entidade `CategoriaIndicador`, enriquecida com o nome em português e uma área temática (agrupamento de categorias definido pelo grupo).
- **Tipo de indicador**: cada indicador recebeu um tipo (percentual, média, taxa, índice ou valor monetário), deduzido do sufixo do código e da descrição.
- **Nomes de região**: duas regiões aparecem com grafias diferentes entre pesquisas (`Hauncavelica`/`Huancavelica` no Peru e `Western  Cape`/`Western Cape` na África do Sul). Foi mantida a grafia da pesquisa mais recente.
- **Códigos de indicador**: alguns códigos usam hífen (`FF_ASFR_15-19`) enquanto a coluna correspondente no CSV usa ponto (`FF_ASFR_15.19`); o mapeamento é feito na filtragem.

## 3.1 Configuração

In [15]:
import sqlite3
from pathlib import Path

import pandas as pd

RAW = Path("data/raw")
PROCESSED = Path("data/processed")
DB_PATH = Path("saudefeminina.db")

# Fecha a conexão de uma execução anterior (no Windows, o arquivo aberto não pode ser apagado)
if "con" in globals():
    con.close()

# O banco é sempre recriado do zero a partir dos CSVs
DB_PATH.unlink(missing_ok=True)
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

## 3.2 Leitura dos dados brutos

In [16]:
livwell = pd.read_csv(RAW / "livwell.csv")
indicadores_raw = pd.read_csv(RAW / "indicators.csv")
selecao = pd.read_csv(PROCESSED / "filtered_indicators.csv")

print("livwell.csv:", livwell.shape)
print("indicators.csv:", indicadores_raw.shape)
print("Indicadores selecionados:", len(selecao))

# Todo indicador selecionado deve existir no dicionário original
assert selecao["indicator_code"].isin(indicadores_raw["indicator_code"]).all()

selecao["indicator_category"].value_counts().to_frame("indicadores")

livwell.csv: (1832, 409)
indicators.csv: (265, 3)
Indicadores selecionados: 100


,indicadores
indicator_category,
Individual demographic information,13
Energy and information,12
Reproductive health and fertility,11
Decision power,10
Domestic Violence,10
Education,9
Fertility – complex indicators,9
Health – Birth level,5
Household characteristics – Womens level,4


## 3.3 Filtragem das colunas

In [17]:
COLUNAS_ID = [
    "country_name", "country_code", "year",
    "region_num_harmonized", "region_name_harmonized",
    "SurveyId", "interview_year_mean", "interview_month_mean", "CMC_interview_mean",
]

# Código do indicador -> coluna de valor e coluna de erro-padrão (se existir)
mapa = {}
for codigo in selecao["indicator_code"]:
    coluna = codigo.replace("-", ".")
    assert coluna in livwell.columns, f"Coluna não encontrada: {coluna}"
    coluna_se = coluna + "_se"
    mapa[codigo] = (coluna, coluna_se if coluna_se in livwell.columns else None)

colunas = COLUNAS_ID + [c for par in mapa.values() for c in par if c]
filtrado = livwell[colunas].copy()

print("Dados filtrados:", filtrado.shape)
print("Indicadores sem coluna de erro-padrão:", sum(se is None for _, se in mapa.values()))

Dados filtrados: (1832, 192)
Indicadores sem coluna de erro-padrão: 17


## 3.4 Criação do esquema

In [18]:
cur.executescript("""
PRAGMA foreign_keys = 0;

CREATE TABLE Pais (
    country_code TEXT PRIMARY KEY,
    country_name TEXT NOT NULL,
    continent    TEXT NOT NULL
);

CREATE TABLE Regiao (
    country_code           TEXT    NOT NULL REFERENCES Pais(country_code),
    region_num_harmonized  INTEGER NOT NULL,
    region_name_harmonized TEXT    NOT NULL,
    PRIMARY KEY (country_code, region_num_harmonized)
);

CREATE TABLE Pesquisa (
    SurveyId           TEXT PRIMARY KEY,
    country_code       TEXT    NOT NULL REFERENCES Pais(country_code),
    year               INTEGER NOT NULL,
    CMC_interview_mean REAL,
    UNIQUE (SurveyId, country_code)
);

CREATE TABLE ColetaPesquisaRegiao (
    SurveyId              TEXT    NOT NULL,
    country_code          TEXT    NOT NULL,
    region_num_harmonized INTEGER NOT NULL,
    interview_year_mean   REAL,
    interview_month_mean  REAL,
    CMC_interview_mean    REAL,
    PRIMARY KEY (SurveyId, country_code, region_num_harmonized),
    FOREIGN KEY (SurveyId, country_code)
        REFERENCES Pesquisa(SurveyId, country_code),
    FOREIGN KEY (country_code, region_num_harmonized)
        REFERENCES Regiao(country_code, region_num_harmonized)
);

CREATE TABLE CategoriaIndicador (
    category_id      INTEGER PRIMARY KEY,
    category_name    TEXT NOT NULL UNIQUE,
    category_name_pt TEXT NOT NULL,
    category_area    TEXT NOT NULL
);

CREATE TABLE Indicador (
    indicator_code        TEXT PRIMARY KEY,
    indicator_description TEXT NOT NULL,
    indicator_type        TEXT NOT NULL,
    category_id           INTEGER NOT NULL REFERENCES CategoriaIndicador(category_id)
);

CREATE TABLE Medicao (
    measurement_id        INTEGER PRIMARY KEY,
    SurveyId              TEXT    NOT NULL REFERENCES Pesquisa(SurveyId),
    country_code          TEXT    NOT NULL,
    region_num_harmonized INTEGER NOT NULL,
    indicator_code        TEXT    NOT NULL REFERENCES Indicador(indicator_code),
    value                 REAL    NOT NULL,
    standard_error        REAL,
    FOREIGN KEY (country_code, region_num_harmonized)
        REFERENCES Regiao(country_code, region_num_harmonized),
    UNIQUE (SurveyId, country_code, region_num_harmonized, indicator_code)
);
""")

con.commit()

## 3.5 Inserção dos dados

### Pais, Regiao, Pesquisa e ColetaPesquisaRegiao

In [19]:
CONTINENTES = {
    "África": ["BDI", "BEN", "BFA", "CIV", "CMR", "COD", "EGY", "ETH", "GAB", "GHA", "GIN",
               "KEN", "LBR", "LSO", "MAR", "MDG", "MLI", "MOZ", "MWI", "NAM", "NER", "NGA",
               "RWA", "SEN", "SLE", "TGO", "TZA", "UGA", "ZAF", "ZMB", "ZWE"],
    "América": ["BOL", "COL", "GTM", "HND", "HTI", "NIC", "PER"],
    "Ásia": ["ARM", "BGD", "IDN", "IND", "JOR", "KHM", "MDV", "NPL", "PAK", "PHL", "TJK",
             "TLS", "TUR", "VNM"],
}
continente_de = {pais: cont for cont, paises in CONTINENTES.items() for pais in paises}

pais = filtrado[["country_code", "country_name"]].drop_duplicates("country_code")
pais["continent"] = pais["country_code"].map(continente_de)
assert pais["continent"].notna().all()
pais = pais.sort_values("country_code")
pais.to_sql("Pais", con, if_exists="append", index=False)

regiao = (
    filtrado.sort_values("year", ascending=False)
    .drop_duplicates(["country_code", "region_num_harmonized"])
    [["country_code", "region_num_harmonized", "region_name_harmonized"]]
)
regiao.to_sql("Regiao", con, if_exists="append", index=False)

assert (filtrado.groupby("SurveyId")[["country_code", "year"]].nunique() == 1).all().all()
pesquisa = filtrado.groupby("SurveyId", as_index=False).agg(
    country_code=("country_code", "first"),
    year=("year", "first"),
    CMC_interview_mean=("CMC_interview_mean", "mean"),
)
pesquisa.to_sql("Pesquisa", con, if_exists="append", index=False)

coleta = filtrado[[
    "SurveyId", "country_code", "region_num_harmonized",
    "interview_year_mean", "interview_month_mean", "CMC_interview_mean",
]]
coleta.to_sql("ColetaPesquisaRegiao", con, if_exists="append", index=False)

print("Países:", len(pais), "| Regiões:", len(regiao),
      "| Pesquisas:", len(pesquisa), "| Pares pesquisa x região:", len(coleta))

Países: 52 | Regiões: 481 | Pesquisas: 199 | Pares pesquisa x região: 1832


### CategoriaIndicador e Indicador

In [20]:
CATEGORIAS = {
    "Decision power": ("Poder de decisão", "Autonomia e violência"),
    "Domestic Violence": ("Violência doméstica", "Autonomia e violência"),
    "Education": ("Educação", "Educação e trabalho"),
    "Work status": ("Situação de trabalho", "Educação e trabalho"),
    "Energy and information": ("Energia e informação", "Domicílio e condições de vida"),
    "Energy and information – per urban/rural area": ("Energia e informação – por área urbana/rural", "Domicílio e condições de vida"),
    "Household characteristics – Womens level": ("Características do domicílio – nível da mulher", "Domicílio e condições de vida"),
    "Household demographics – Household level": ("Demografia do domicílio – nível do domicílio", "Domicílio e condições de vida"),
    "Household Wealth": ("Riqueza do domicílio", "Domicílio e condições de vida"),
    "Socio-economic indicators": ("Indicadores socioeconômicos", "Domicílio e condições de vida"),
    "Individual demographic information": ("Informações demográficas individuais", "Demografia"),
    "Fertility – complex indicators": ("Fecundidade – indicadores compostos", "Saúde reprodutiva"),
    "Fertility preferences": ("Preferências de fecundidade", "Saúde reprodutiva"),
    "Reproductive health and fertility": ("Saúde reprodutiva e fecundidade", "Saúde reprodutiva"),
    "Health – Birth level": ("Saúde – nível do nascimento", "Saúde infantil e nutrição"),
    "Nutrition": ("Nutrição", "Saúde infantil e nutrição"),
}
assert set(selecao["indicator_category"]) == set(CATEGORIAS)

categoria = pd.DataFrame(
    [(nome, pt, area) for nome, (pt, area) in CATEGORIAS.items()],
    columns=["category_name", "category_name_pt", "category_area"],
)
categoria.insert(0, "category_id", range(1, len(categoria) + 1))
categoria.to_sql("CategoriaIndicador", con, if_exists="append", index=False)


def tipo_indicador(codigo):
    """Tipo de medida, deduzido do sufixo do código e da descrição do indicador."""
    if codigo.endswith("_p"):
        return "Percentual"
    if codigo.endswith("_mean"):
        return "Média"
    if codigo.startswith(("FF_", "HL_")):
        return "Taxa"
    if codigo == "gdp_pc":
        return "Valor monetário"
    return "Índice"


indicador = selecao.merge(categoria, left_on="indicator_category", right_on="category_name")
indicador["indicator_type"] = indicador["indicator_code"].map(tipo_indicador)
indicador = indicador[["indicator_code", "indicator_description", "indicator_type", "category_id"]]
indicador.to_sql("Indicador", con, if_exists="append", index=False)

print("Categorias:", len(categoria), "| Indicadores:", len(indicador))
indicador["indicator_type"].value_counts().to_frame("indicadores")

Categorias: 16 | Indicadores: 100


,indicadores
indicator_type,
Percentual,67
Média,15
Taxa,14
Índice,3
Valor monetário,1


### Medicao

Cada coluna de indicador vira um conjunto de linhas (formato largo → longo).

In [21]:
chaves = ["SurveyId", "country_code", "region_num_harmonized"]
partes = []
for codigo, (coluna, coluna_se) in mapa.items():
    parte = filtrado[chaves].copy()
    parte["indicator_code"] = codigo
    parte["value"] = filtrado[coluna]
    parte["standard_error"] = filtrado[coluna_se] if coluna_se else None
    partes.append(parte)

medicao = pd.concat(partes, ignore_index=True).dropna(subset=["value"])
medicao.to_sql("Medicao", con, if_exists="append", index=False)
con.commit()

print("Medições:", len(medicao))

Medições: 147829


## 3.6 Criação de regras de exclusão

In [22]:
# Ativa integridade referencial nesta conexão
cur.execute("PRAGMA foreign_keys = ON")

# Gatilhos para impedir exclusões de entidades compartilhadas e que ainda possuem registros dependentes.
restricoes = {
    "Pais": [
        ("Regiao", "country_code = OLD.country_code"),
        ("Pesquisa", "country_code = OLD.country_code"),
    ],
    "Regiao": [
        (
            "ColetaPesquisaRegiao",
            """country_code = OLD.country_code
               AND region_num_harmonized = OLD.region_num_harmonized"""
        ),
        (
            "Medicao",
            """country_code = OLD.country_code
               AND region_num_harmonized = OLD.region_num_harmonized"""
        ),
    ],
    "CategoriaIndicador": [
        ("Indicador", "category_id = OLD.category_id"),
    ],
    "Indicador": [
        ("Medicao", "indicator_code = OLD.indicator_code"),
    ],
}

for tabela_pai, dependencias in restricoes.items():
    for tabela_filha, condicao in dependencias:
        nome = f"trg_restrict_{tabela_pai}_{tabela_filha}"

        cur.execute(f"""
            CREATE TRIGGER IF NOT EXISTS {nome}
            BEFORE DELETE ON {tabela_pai}
            FOR EACH ROW
            WHEN EXISTS (
                SELECT 1
                FROM {tabela_filha}
                WHERE {condicao}
            )
            BEGIN
                SELECT RAISE(
                    ABORT,
                    'Exclusao bloqueada: existem registros dependentes em {tabela_filha}'
                );
            END;
        """)

# CASCADE: excluir as coletas regionais de uma pesquisa quando a própria pesquisa for excluída.
cur.execute("""
    CREATE TRIGGER IF NOT EXISTS trg_cascade_Pesquisa_Coleta
    BEFORE DELETE ON Pesquisa
    FOR EACH ROW
    BEGIN
        DELETE FROM ColetaPesquisaRegiao
        WHERE SurveyId = OLD.SurveyId
          AND country_code = OLD.country_code;
    END;
""")

# CASCADE: excluir as medições de uma pesquisa quando a própria pesquisa for excluída.
cur.execute("""
    CREATE TRIGGER IF NOT EXISTS trg_cascade_Pesquisa_Medicao
    BEFORE DELETE ON Pesquisa
    FOR EACH ROW
    BEGIN
        DELETE FROM Medicao
        WHERE SurveyId = OLD.SurveyId;
    END;
""")

con.commit()

# Lista os gatilhos instalados
gatilhos = cur.execute("""
    SELECT name, tbl_name, sql
    FROM sqlite_master
    WHERE type = 'trigger'
    ORDER BY name
""").fetchall()

for nome, tabela, definicao in gatilhos:
    print(f"\n{nome} — tabela: {tabela}")


trg_cascade_Pesquisa_Coleta — tabela: Pesquisa

trg_cascade_Pesquisa_Medicao — tabela: Pesquisa

trg_restrict_CategoriaIndicador_Indicador — tabela: CategoriaIndicador

trg_restrict_Indicador_Medicao — tabela: Indicador

trg_restrict_Pais_Pesquisa — tabela: Pais

trg_restrict_Pais_Regiao — tabela: Pais

trg_restrict_Regiao_ColetaPesquisaRegiao — tabela: Regiao

trg_restrict_Regiao_Medicao — tabela: Regiao


## 3.7 Verificação

In [23]:
cur.execute("PRAGMA foreign_keys = 1")
erros_fk = cur.execute("PRAGMA foreign_key_check").fetchall()
print("Erros de integridade referencial:", erros_fk)
assert not erros_fk

orfas = cur.execute("""
    SELECT COUNT(*) FROM Medicao m
    LEFT JOIN ColetaPesquisaRegiao a USING (SurveyId, country_code, region_num_harmonized)
    WHERE a.SurveyId IS NULL
""").fetchone()[0]
print("Medições sem par em ColetaPesquisaRegiao:", orfas)
assert orfas == 0

tabelas = ["Pais", "Regiao", "Pesquisa", "ColetaPesquisaRegiao", "CategoriaIndicador", "Indicador", "Medicao"]
pd.DataFrame(
    {"registros": [cur.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tabelas]},
    index=tabelas,
)

Erros de integridade referencial: []
Medições sem par em ColetaPesquisaRegiao: 0


,registros
Pais,52
Regiao,481
Pesquisa,199
ColetaPesquisaRegiao,1832
CategoriaIndicador,16
Indicador,100
Medicao,147829


In [24]:
# Amostra do banco normalizado
pd.read_sql_query("""
    SELECT pa.country_name, pa.continent, r.region_name_harmonized,
           p.SurveyId, p.year, a.interview_month_mean,
           c.category_name_pt, i.indicator_code, i.indicator_type,
           m.value, m.standard_error
    FROM Medicao m
    JOIN ColetaPesquisaRegiao a   USING (SurveyId, country_code, region_num_harmonized)
    JOIN Regiao r    USING (country_code, region_num_harmonized)
    JOIN Pais pa     USING (country_code)
    JOIN Pesquisa p  USING (SurveyId)
    JOIN Indicador i USING (indicator_code)
    JOIN CategoriaIndicador c USING (category_id)
    WHERE i.indicator_code = 'RH_contr_modern_p'
    LIMIT 5
""", con)

,country_name,continent,region_name_harmonized,SurveyId,year,interview_month_mean,category_name_pt,indicator_code,indicator_type,value,standard_error
0,Armenia,Ásia,Aragatsotn,AM2000DHS,2000,11.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,10.12,0.97
1,Armenia,Ásia,Ararat,AM2000DHS,2000,11.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,18.26,1.90
2,Armenia,Ásia,Armavir,AM2000DHS,2000,10.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,12.53,1.11
3,Armenia,Ásia,Gegharkunik,AM2000DHS,2000,11.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,13.09,1.96
4,Armenia,Ásia,Lori,AM2000DHS,2000,10.0,Saúde reprodutiva e fecundidade,RH_contr_modern_p,Percentual,14.43,1.56


# 4. Diagrama ER

Para ajudar no entendimento das relações do sistema desenvolvido, foi construído um diagrama que pode ser acessado no link abaixo.

https://lucid.app/lucidchart/74952537-0644-48d7-9527-9f8bb02de887/edit?viewport_loc=-2663%2C-1030%2C5760%2C2562%2C0_0&invitationId=inv_f3bf9b73-e139-47f8-99a3-6d149bb52dd9

![Diagrama ER](diagramas/diagrama_er.png)

# 5. Diagrama relacional

O diagrama relacional mostra as sete tabelas do banco, com suas chaves primárias (PK), chaves estrangeiras (FK) e as ligações entre elas:

![Diagrama relacional](diagramas/diagrama_relacional.png)

# 6. Consultas

## 6.1 Duas consultas envolvendo seleção e projeção

### 6.1.1 Consulta 1

### Consulta 1: violência do parceiro em 40% ou mais das mulheres

Seleção e projeção: seleciona as medições regionais em que 40% ou mais das mulheres sofreram violência física ou sexual do parceiro nos últimos 12 meses, com o erro-padrão de cada valor.

**Interpretação:** São 87 medições, de 15 países. O maior valor é 78,09%, numa região do Congo em 2007. Peru (16 linhas) e Congo (15) são os países que mais aparecem, então o problema não se limita à África. O erro-padrão mostra a incerteza de cada valor (4,79 pontos no maior).

In [25]:
query = """
SELECT SurveyId, country_code, region_num_harmonized,
       value, standard_error
FROM Medicao
WHERE indicator_code = 'DV_phys_or_sex_partner_12m_p'
  AND value >= 40
ORDER BY value DESC
"""
resultado = pd.read_sql_query(query, con)
resultado

,SurveyId,country_code,region_num_harmonized,value,standard_error
0,CD2007DHS,COD,4,78.09,4.79
1,TL2009DHS,TLS,11,73.68,4.51
2,CD2007DHS,COD,5,69.76,6.67
3,CD2007DHS,COD,2,64.29,3.93
4,KE2003DHS,KEN,7,63.03,2.92
...,...,...,...,...,...
82,IA2006DHS,IND,4,40.76,1.53
83,CD2013DHS,COD,4,40.73,2.81
84,TZ2010DHS,TZA,16,40.68,4.76
85,PE2000DHS,PER,10,40.25,2.08


### 6.1.2 Consulta 2

### Consulta 2: regiões com uso quase nulo de contracepção moderna

Seleção e projeção: seleciona as medições regionais em que menos de 2% das mulheres usam método contraceptivo moderno.

**Interpretação:** São 20 medições: 19 em países africanos e 1 no Paquistão. O Níger aparece 5 vezes, e 11 das 20 são de pesquisas anteriores a 2000.

In [26]:
query = """
SELECT SurveyId, country_code, region_num_harmonized, value
FROM Medicao
WHERE indicator_code = 'RH_contr_modern_p'
  AND value < 2
ORDER BY value
"""
resultado = pd.read_sql_query(query, con)
resultado

,SurveyId,country_code,region_num_harmonized,value
0,KE2003DHS,KEN,8,0.12
1,MZ1997DHS,MOZ,2,0.58
2,ET2016DHS,ETH,5,0.95
3,MD1992DHS,MDG,2,0.99
4,NI1992DHS,NER,4,1.10
5,NI1992DHS,NER,2,1.11
6,NI1992DHS,NER,3,1.23
7,ML1996DHS,MLI,5,1.32
8,PK1991DHS,PAK,4,1.41
9,ML2001DHS,MLI,7,1.42


## 6.2 Três consultas envolvendo junção de duas relações

### 6.2.1 Consulta 3

### Consulta 3: coletas feitas em ano diferente do ano da pesquisa

Junção de duas relações: combina ColetaPesquisaRegiao e Pesquisa para comparar o ano médio das entrevistas em cada região com o ano da pesquisa.

**Interpretação:** Em 229 das 1832 coletas regionais, o ano das entrevistas é diferente do ano da pesquisa. Na PE2004DHS, por exemplo, há regiões entrevistadas em 2006 e outras em 2007. Por isso a data de coleta fica no relacionamento entre pesquisa e região, e não na Pesquisa. As 44 linhas da Etiópia, com diferença de -8 anos, são uma inconsistência da fonte.

In [27]:
query = """
SELECT c.SurveyId, p.year AS ano_pesquisa, c.region_num_harmonized,
       ROUND(c.interview_year_mean, 2) AS ano_medio_coleta,
       ROUND(c.interview_month_mean, 1) AS mes_medio_coleta,
       ROUND(c.interview_year_mean) - p.year AS diferenca_anos
FROM ColetaPesquisaRegiao AS c
JOIN Pesquisa AS p
  ON p.SurveyId = c.SurveyId
 AND p.country_code = c.country_code
WHERE ROUND(c.interview_year_mean) <> p.year
ORDER BY ABS(diferenca_anos) DESC, c.SurveyId, c.region_num_harmonized
"""
resultado = pd.read_sql_query(query, con)
resultado

,SurveyId,ano_pesquisa,region_num_harmonized,ano_medio_coleta,mes_medio_coleta,diferenca_anos
0,ET2000DHS,2000,1,1992.0,8.0,-8.0
1,ET2000DHS,2000,2,1992.0,8.0,-8.0
2,ET2000DHS,2000,3,1992.0,7.0,-8.0
3,ET2000DHS,2000,4,1992.0,7.0,-8.0
4,ET2000DHS,2000,5,1992.0,8.0,-8.0
...,...,...,...,...,...,...
224,TG2013DHS,2013,6,2014.0,2.0,1.0
225,TR2003DHS,2003,3,2004.0,1.0,1.0
226,TR2003DHS,2003,4,2004.0,2.0,1.0
227,TR2003DHS,2003,5,2004.0,3.0,1.0


### 6.2.2 Consulta 4

### Consulta 4: regiões com maior fecundidade na adolescência

Junção de duas relações: combina Medicao e Regiao para mostrar o nome das regiões com mais nascimentos por mil mulheres de 15 a 19 anos.

**Interpretação:** A maior taxa é de cerca de 297 nascimentos por mil adolescentes por ano, em Maradi (Níger, 1998). Maradi e Zinder/Diffa ocupam 6 das 10 linhas, em pesquisas de 1992 a 2012: a fecundidade adolescente continua alta nessas regiões ao longo do tempo.

In [28]:
query = """
SELECT r.region_name_harmonized, m.country_code, m.SurveyId,
       m.value AS nascimentos_por_mil_15_19
FROM Medicao AS m
JOIN Regiao AS r
  ON r.country_code = m.country_code
 AND r.region_num_harmonized = m.region_num_harmonized
WHERE m.indicator_code = 'FF_ASFR_15-19'
ORDER BY m.value DESC
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

,region_name_harmonized,country_code,SurveyId,nascimentos_por_mil_15_19
0,Maradi,NER,NI1998DHS,296.858
1,Zinder /Diffa,NER,NI1998DHS,292.616
2,Niassa,MOZ,MZ2003DHS,270.243
3,Maradi,NER,NI2006DHS,265.822
4,Zinder /Diffa,NER,NI2012DHS,265.228
5,Pwani,TZA,TZ1999DHS,263.067
6,Zinder /Diffa,NER,NI2006DHS,258.574
7,Mara,TZA,TZ1996DHS,258.126
8,Upper Egypt,EGY,EG1992DHS,257.068
9,Zinder /Diffa,NER,NI1992DHS,256.670


### 6.2.3 Consulta 5

### Consulta 5: indicadores de autonomia e violência

Junção de duas relações: associa cada indicador à sua categoria e lista os da área temática Autonomia e violência.

**Interpretação:** A base tem 20 indicadores dessa área, 10 de poder de decisão e 10 de violência doméstica, todos percentuais. Eles permitem relacionar o planejamento familiar à autonomia das mulheres, como na consulta 9.

In [29]:
query = """
SELECT c.category_name_pt, i.indicator_code,
       i.indicator_description, i.indicator_type
FROM Indicador AS i
JOIN CategoriaIndicador AS c USING (category_id)
WHERE c.category_area = 'Autonomia e violência'
ORDER BY c.category_name_pt, i.indicator_code
"""
resultado = pd.read_sql_query(query, con)
resultado

,category_name_pt,indicator_code,indicator_description,indicator_type
0,Poder de decisão,DP_decide_contraception_p,Women deciding alone on their contraception (%),Percentual
1,Poder de decisão,DP_decide_health_p,Women who have the final say on their health c...,Percentual
2,Poder de decisão,DP_decide_large_purchase_p,Women who have the final say on large househol...,Percentual
3,Poder de decisão,DP_decide_money_p,Women currently married or in union who were p...,Percentual
4,Poder de decisão,DP_decide_no_contraception_p,Women deciding alone on not using contraceptio...,Percentual
5,Poder de decisão,DP_decide_visits_p,Women who have the final say on visiting famil...,Percentual
6,Poder de decisão,DP_earn_more_equal_p,Women currently married or in union who were ...,Percentual
7,Poder de decisão,DP_earn_more_p,Women currently married or in union who were ...,Percentual
8,Poder de decisão,DP_owns_house_p,Women owning a house alone or jointly (%),Percentual
9,Poder de decisão,DP_owns_land_p,Women owning land alone or jointly (%),Percentual


## 6.3 Três consultas envolvendo junção de três ou mais relações

### 6.3.1 Consulta 6

### Consulta 6: mulheres que conhecem métodos modernos mas não usam

Junção de três ou mais relações: compara, na mesma pesquisa e região, o conhecimento e o uso de métodos modernos (duas cópias de Medicao), com Regiao, Pesquisa e Pais, a partir de 2010.

**Interpretação:** Nas 10 regiões com maior diferença, mais de 95% das mulheres conhecem algum método moderno, mas menos de 9% usam. O baixo uso não se explica por falta de conhecimento; a consulta não mostra qual é a barreira.

In [30]:
query = """
SELECT pa.country_name, r.region_name_harmonized, p.year,
       k.value AS conhece_metodo_moderno,
       u.value AS usa_metodo_moderno,
       ROUND(k.value - u.value, 2) AS lacuna
FROM Medicao AS k
JOIN Medicao AS u
  ON u.SurveyId = k.SurveyId
 AND u.country_code = k.country_code
 AND u.region_num_harmonized = k.region_num_harmonized
JOIN Regiao AS r
  ON r.country_code = k.country_code
 AND r.region_num_harmonized = k.region_num_harmonized
JOIN Pesquisa AS p ON p.SurveyId = k.SurveyId
JOIN Pais AS pa ON pa.country_code = k.country_code
WHERE k.indicator_code = 'RH_contr_modern_know_p'
  AND u.indicator_code = 'RH_contr_modern_p'
  AND p.year >= 2010
ORDER BY lacuna DESC
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

,country_name,region_name_harmonized,year,conhece_metodo_moderno,usa_metodo_moderno,lacuna
0,Mozambique,Cabo Delgado,2011,99.65,3.26,96.39
1,Tanzania,Pemba North,2010,97.53,3.86,93.67
2,Armenia,Tavush,2010,99.39,6.84,92.55
3,Tanzania,Zanzibar North,2010,96.88,4.44,92.44
4,Mozambique,Zambezia,2011,96.87,5.09,91.78
5,Armenia,Lori,2016,99.60,8.09,91.51
6,Burundi,Bubanza + Bujumbura Rural + Bururi + Cibitoke ...,2010,97.16,6.38,90.78
7,Burkina Faso,Centre-Nord,2010,98.65,8.21,90.44
8,Tanzania,Zanzibar town or West,2015,98.82,8.43,90.39
9,Tanzania,Pemba South,2010,95.79,5.68,90.11


### 6.3.2 Consulta 7

### Consulta 7: regiões onde as mulheres têm mais filhos do que o ideal

Junção de três ou mais relações: compara o número ideal de filhos declarado (FP_desired_mean) com a taxa de fecundidade total (FF_TFR), na mesma pesquisa e região, a partir de 2010.

**Interpretação:** Há regiões onde as mulheres têm cerca de 2 filhos a mais do que o ideal. O Burundi ocupa 5 das 10 linhas, e nas duas regiões dele que aparecem em 2010 e 2016 a diferença diminuiu. A comparação é aproximada: o ideal é declarado e a TFR é uma taxa do período.

In [31]:
query = """
SELECT pa.country_name, r.region_name_harmonized, p.year,
       d.value AS filhos_ideal,
       t.value AS filhos_tfr,
       ROUND(t.value - d.value, 2) AS diferenca
FROM Medicao AS d
JOIN Medicao AS t
  ON t.SurveyId = d.SurveyId
 AND t.country_code = d.country_code
 AND t.region_num_harmonized = d.region_num_harmonized
JOIN Regiao AS r
  ON r.country_code = d.country_code
 AND r.region_num_harmonized = d.region_num_harmonized
JOIN Pesquisa AS p ON p.SurveyId = d.SurveyId
JOIN Pais AS pa ON pa.country_code = d.country_code
WHERE d.indicator_code = 'FP_desired_mean'
  AND t.indicator_code = 'FF_TFR'
  AND p.year >= 2010
ORDER BY diferenca DESC
LIMIT 10
"""
resultado = pd.read_sql_query(query, con)
resultado

,country_name,region_name_harmonized,year,filhos_ideal,filhos_tfr,diferenca
0,Burundi,Kayanza + Kirundo + Muyinga + Ngozi,2010,3.92,6.690,2.77
1,Uganda,Bugosa + Bukedi + Bugishu + Teso,2011,4.95,7.264,2.31
2,Mozambique,Tete,2011,4.57,6.817,2.25
3,Burundi,Cankuzo + Gitega + Karusi + Muramvya + Ruyigi,2010,4.15,6.332,2.18
4,Peru,Loreto,2011,2.43,4.575,2.15
5,Burundi,Bubanza + Bujumbura Rural + Bururi + Cibitoke ...,2010,4.45,6.575,2.13
6,Tanzania,Katavi + Rukwa,2010,6.01,8.012,2.00
7,Tanzania,Mara,2010,5.14,7.127,1.99
8,Burundi,Kayanza + Kirundo + Muyinga + Ngozi,2016,3.62,5.558,1.94
9,Burundi,Cankuzo + Gitega + Karusi + Muramvya + Ruyigi,2016,3.81,5.719,1.91


### 6.3.3 Consulta 8

### Consulta 8: escolaridade e contracepção moderna nas regiões da Nigéria

Junção de três ou mais relações: compara, em cada região da Nigéria em 2018, o percentual de mulheres com ensino secundário ou mais e o uso de contracepção moderna.

**Interpretação:** As três regiões do Sul, com 77% a 80% de mulheres escolarizadas, têm de 10,6% a 18,4% de uso. As duas do Norte com menor escolaridade (25% a 28%) têm os menores usos (5,3% e 6,9%). A relação não é perfeita: South East tem mais escolaridade que North Central e uso menor.

In [32]:
query = """
SELECT pa.country_name, r.region_name_harmonized, p.year,
       e.value AS pct_secundario_ou_mais,
       c.value AS pct_contracepcao_moderna
FROM Medicao AS e
JOIN Medicao AS c
  ON c.SurveyId = e.SurveyId
 AND c.country_code = e.country_code
 AND c.region_num_harmonized = e.region_num_harmonized
JOIN Regiao AS r
  ON r.country_code = e.country_code
 AND r.region_num_harmonized = e.region_num_harmonized
JOIN Pesquisa AS p ON p.SurveyId = e.SurveyId
JOIN Pais AS pa ON pa.country_code = e.country_code
WHERE e.indicator_code = 'ED_attainment_secondary_higher_p'
  AND c.indicator_code = 'RH_contr_modern_p'
  AND pa.country_code = 'NGA'
  AND p.year = 2018
ORDER BY e.value DESC
"""
resultado = pd.read_sql_query(query, con)
resultado

,country_name,region_name_harmonized,year,pct_secundario_ou_mais,pct_contracepcao_moderna
0,Nigeria,South South,2018,79.55,15.47
1,Nigeria,South West,2018,77.50,18.38
2,Nigeria,South East,2018,77.25,10.58
3,Nigeria,North Central,2018,49.85,11.46
4,Nigeria,North East,2018,28.16,6.94
5,Nigeria,North West,2018,25.03,5.33


## 6.4 Duas consultas envolvendo agregação sobre junção de duas ou mais relações

### 6.4.1 Consulta 9

### Consulta 9: autonomia e contracepção moderna por continente

Agregação sobre junção de duas cópias de Medicao e Pais: agrupa as medições por continente e pela faixa de mulheres que decidem sobre a própria saúde, e calcula o uso médio de contracepção moderna.

**Interpretação:** Nos três continentes, o uso médio cresce a cada faixa de autonomia. Na África vai de 11,9% para 29,7%, na América de 23,4% para 35,5% e na Ásia de 22,6% para 28,3%. É uma associação, não uma relação de causa.

In [33]:
query = """
SELECT pa.continent,
       CASE
         WHEN a.value < 10 THEN '1) 0 a 10%'
         WHEN a.value < 20 THEN '2) 10 a 20%'
         WHEN a.value < 30 THEN '3) 20 a 30%'
         ELSE '4) 30% ou mais'
       END AS faixa_autonomia_saude,
       COUNT(*) AS medicoes_regionais,
       ROUND(AVG(c.value), 1) AS media_contracepcao
FROM Medicao AS a
JOIN Medicao AS c
  ON c.SurveyId = a.SurveyId
 AND c.country_code = a.country_code
 AND c.region_num_harmonized = a.region_num_harmonized
JOIN Pais AS pa ON pa.country_code = a.country_code
WHERE a.indicator_code = 'DP_decide_health_p'
  AND c.indicator_code = 'RH_contr_modern_p'
GROUP BY pa.continent, faixa_autonomia_saude
ORDER BY pa.continent, faixa_autonomia_saude
"""
resultado = pd.read_sql_query(query, con)
resultado

,continent,faixa_autonomia_saude,medicoes_regionais,media_contracepcao
0,América,2) 10 a 20%,18,23.4
1,América,3) 20 a 30%,39,31.9
2,América,4) 30% ou mais,243,35.5
3,África,1) 0 a 10%,136,11.9
4,África,2) 10 a 20%,205,18.3
5,África,3) 20 a 30%,131,21.8
6,África,4) 30% ou mais,166,29.7
7,Ásia,1) 0 a 10%,33,22.6
8,Ásia,2) 10 a 20%,75,24.7
9,Ásia,3) 20 a 30%,64,27.6


### 6.4.2 Consulta 10

### Consulta 10: contracepção moderna por faixa de escolaridade

Agregação sobre junção de duas cópias de Medicao: agrupa as medições pela faixa de mulheres com ensino secundário ou mais e calcula média, mínimo e máximo do uso de contracepção moderna.

**Interpretação:** O uso médio quase dobra entre a faixa de menor escolaridade (16,1%) e a de 50% a 75% (31,3%). Acima de 75% ele para de subir (30,0%). Os dados são por região, não por mulher.

In [34]:
query = """
SELECT CASE
         WHEN e.value < 25 THEN '1) 0 a 25%'
         WHEN e.value < 50 THEN '2) 25 a 50%'
         WHEN e.value < 75 THEN '3) 50 a 75%'
         ELSE '4) 75 a 100%'
       END AS faixa_escolaridade,
       COUNT(*) AS medicoes_regionais,
       ROUND(AVG(c.value), 1) AS media_contracepcao,
       ROUND(MIN(c.value), 1) AS minimo,
       ROUND(MAX(c.value), 1) AS maximo
FROM Medicao AS e
JOIN Medicao AS c
  ON c.SurveyId = e.SurveyId
 AND c.country_code = e.country_code
 AND c.region_num_harmonized = e.region_num_harmonized
WHERE e.indicator_code = 'ED_attainment_secondary_higher_p'
  AND c.indicator_code = 'RH_contr_modern_p'
GROUP BY faixa_escolaridade
ORDER BY faixa_escolaridade
"""
resultado = pd.read_sql_query(query, con)
resultado

,faixa_escolaridade,medicoes_regionais,media_contracepcao,minimo,maximo
0,1) 0 a 25%,667,16.1,0.1,49.5
1,2) 25 a 50%,528,27.4,1.7,56.9
2,3) 50 a 75%,415,31.3,4.6,60.1
3,4) 75 a 100%,214,30.0,6.8,65.7


# 7. Autoavaliação dos membros

**Desenvolvimento do trabalho pelo grupo**

O grupo procurou uma base de dados completa sobre saúde feminina que permitisse explorar um tema específico. Encontramos a base LivWell no Kaggle e, a partir dela, montamos a proposta do trabalho com a descrição do conjunto de dados.

No início, pensamos em usar todos os indicadores da base. Percebemos, porém, que eles tratavam de muitos tópicos diferentes, e por isso fizemos uma pré-seleção de indicadores e a normalização dos dados. As regras do modelo e a construção dos diagramas ER e relacional foram definidas a partir de discussões do grupo, tanto em sala de aula (laboratório) quanto online, pelo grupo de WhatsApp.

Depois de montar os diagramas e o banco, escolhemos as consultas de acordo com os tópicos que gostaríamos de abordar e com a linha que queríamos seguir na apresentação. Nesse momento, percebemos que poderíamos ter selecionado ainda menos indicadores, já que as consultas não usariam nem metade deles, algo que não havíamos considerado inicialmente. Por isso, revisamos o tema para deixá-lo mais enxuto: sem esse ajuste, teríamos que tratar de muitos assuntos ao mesmo tempo. Consideramos que essa revisão faz parte do processo. Em uma próxima etapa, pretendemos reduzir a seleção de indicadores e remover as colunas desnecessárias da base, para processar os dados de forma ainda mais adequada.